# 🐕 Module 3: Deep Learning for Dog Breed Recognition

**120-Breed Classification | Transfer Learning | YOLOv8 | Grad-CAM**

---
## 📦 Section 1: Setup & Environment

In [ ]:
!pip install -q ultralytics grad-cam

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
import torchvision
from torchvision import datasets, transforms, models
import numpy as np
import matplotlib.pyplot as plt
import cv2
import os
from PIL import Image
from tqdm import tqdm
from google.colab import drive, files
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Using device: {device}')
if torch.cuda.is_available():
    print(f'   GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
drive.mount('/content/drive')
DATA_PATH = '/content/drive/MyDrive/Stanford_Dogs/images'
print(f'✅ Drive mounted')

---
## 📊 Section 2: Dataset Loading (120 Breeds)

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

breeds = sorted(os.listdir(DATA_PATH))
NUM_CLASSES = len(breeds)
print(f'✅ Found {NUM_CLASSES} breeds')

total_images = sum(len(os.listdir(os.path.join(DATA_PATH, b))) for b in breeds)
print(f'✅ Total images: {total_images}')

---
## 🔄 Section 3: Data Augmentation & Regularization

In [ ]:
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    transforms.RandomErasing(p=0.1)
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

print('✅ Transforms defined')

In [ ]:
full_dataset = datasets.ImageFolder(DATA_PATH, transform=train_transform)
class_names = full_dataset.classes

n = len(full_dataset)
n_train, n_val = int(0.7 * n), int(0.15 * n)
n_test = n - n_train - n_val

train_set, val_set, test_set = random_split(full_dataset, [n_train, n_val, n_test])

BATCH_SIZE = 32
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f'✅ Train: {n_train} | Val: {n_val} | Test: {n_test}')

---
## 🧠 Section 4: Transfer Learning CNN (ResNet50)

In [ ]:
# Load pre-trained ResNet50
model = models.resnet50(weights='IMAGENET1K_V2')

# Freeze early layers (conv1 to layer3)
for name, param in model.named_parameters():
    if 'layer4' not in name and 'fc' not in name:
        param.requires_grad = False

# Replace classifier head
model.fc = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(2048, 512),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(512, NUM_CLASSES)
)

model = model.to(device)

# Count trainable params
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'✅ Model ready: {trainable:,} / {total:,} trainable params')

In [ ]:
# Training setup
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4, weight_decay=0.01)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)

print('✅ Optimizer: AdamW | Loss: CrossEntropy with label smoothing')

In [ ]:
# Training loop
NUM_EPOCHS = 15
best_acc = 0
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

for epoch in range(NUM_EPOCHS):
    # Training phase
    model.train()
    train_loss, train_correct = 0, 0
    for images, labels in tqdm(train_loader, desc=f'Epoch {epoch+1}/{NUM_EPOCHS}'):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        train_correct += (outputs.argmax(1) == labels).sum().item()
    
    # Validation phase
    model.eval()
    val_loss, val_correct = 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            val_loss += criterion(outputs, labels).item()
            val_correct += (outputs.argmax(1) == labels).sum().item()
    
    # Metrics
    train_acc = 100 * train_correct / n_train
    val_acc = 100 * val_correct / n_val
    history['train_loss'].append(train_loss / len(train_loader))
    history['val_loss'].append(val_loss / len(val_loader))
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    
    print(f'Train Acc: {train_acc:.1f}% | Val Acc: {val_acc:.1f}%')
    
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), 'best_model.pth')
    
    scheduler.step()

print(f'\n✅ Training complete! Best Val Acc: {best_acc:.1f}%')

In [ ]:
# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history['train_loss'], label='Train')
ax1.plot(history['val_loss'], label='Val')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.legend(); ax1.set_title('Loss')
ax2.plot(history['train_acc'], label='Train')
ax2.plot(history['val_acc'], label='Val')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy %'); ax2.legend(); ax2.set_title('Accuracy')
plt.tight_layout()
plt.show()

In [ ]:
# Test evaluation
model.load_state_dict(torch.load('best_model.pth'))
model.eval()

test_correct, top5_correct = 0, 0
all_preds, all_labels = [], []

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc='Testing'):
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, preds = outputs.topk(5, 1)
        test_correct += (preds[:, 0] == labels).sum().item()
        top5_correct += sum(labels[i] in preds[i] for i in range(len(labels)))
        all_preds.extend(preds[:, 0].cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print(f'\n✅ Test Results:')
print(f'   Top-1 Accuracy: {100*test_correct/n_test:.1f}%')
print(f'   Top-5 Accuracy: {100*top5_correct/n_test:.1f}%')

---
## 🎯 Section 5: Object Detection (YOLOv8)

In [ ]:
from ultralytics import YOLO

# Load YOLOv8 (pre-trained on COCO, includes 'dog' class)
yolo_model = YOLO('yolov8n.pt')
print('✅ YOLOv8 loaded')

In [ ]:
def detect_and_crop_dog(image_path):
    """Detect dog in image and return cropped region"""
    results = yolo_model(image_path, verbose=False)
    img = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    for r in results:
        for box in r.boxes:
            if int(box.cls[0]) == 16:  # Dog class in COCO
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                cropped = img_rgb[y1:y2, x1:x2]
                return cropped, (x1, y1, x2, y2)
    return img_rgb, None

# Demo with sample image
sample_path = os.path.join(DATA_PATH, breeds[0], os.listdir(os.path.join(DATA_PATH, breeds[0]))[0])
cropped, bbox = detect_and_crop_dog(sample_path)
if bbox:
    print(f'✅ Dog detected at {bbox}')
    plt.imshow(cropped)
    plt.title('Detected Dog Region')
    plt.axis('off')
    plt.show()

---
## 🔍 Section 6: Explainability (Grad-CAM + Saliency)

In [ ]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        self._register_hooks()
    
    def _register_hooks(self):
        def forward_hook(module, input, output):
            self.activations = output.detach()
        def backward_hook(module, grad_in, grad_out):
            self.gradients = grad_out[0].detach()
        self.target_layer.register_forward_hook(forward_hook)
        self.target_layer.register_full_backward_hook(backward_hook)
    
    def generate(self, input_tensor, class_idx=None):
        self.model.eval()
        output = self.model(input_tensor)
        if class_idx is None:
            class_idx = output.argmax(1).item()
        
        self.model.zero_grad()
        output[0, class_idx].backward()
        
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        cam = torch.relu(cam)
        cam = cam - cam.min()
        cam = cam / (cam.max() + 1e-8)
        return cam.squeeze().cpu().numpy(), class_idx

gradcam = GradCAM(model, model.layer4[-1])
print('✅ Grad-CAM initialized')

In [ ]:
def visualize_gradcam(image_path):
    """Generate and display Grad-CAM visualization"""
    img = Image.open(image_path).convert('RGB')
    input_tensor = val_transform(img).unsqueeze(0).to(device)
    
    cam, pred_idx = gradcam.generate(input_tensor)
    cam_resized = cv2.resize(cam, (224, 224))
    
    # Overlay heatmap
    img_np = np.array(img.resize((224, 224))) / 255.0
    heatmap = plt.cm.jet(cam_resized)[:, :, :3]
    overlay = 0.5 * img_np + 0.5 * heatmap
    
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(img_np); axes[0].set_title('Original'); axes[0].axis('off')
    axes[1].imshow(cam_resized, cmap='jet'); axes[1].set_title('Grad-CAM'); axes[1].axis('off')
    axes[2].imshow(overlay); axes[2].set_title(f'Pred: {class_names[pred_idx]}'); axes[2].axis('off')
    plt.tight_layout()
    plt.show()

# Demo
visualize_gradcam(sample_path)

In [ ]:
def compute_saliency(image_path):
    """Compute saliency map using input gradients"""
    img = Image.open(image_path).convert('RGB')
    input_tensor = val_transform(img).unsqueeze(0).to(device)
    input_tensor.requires_grad = True
    
    model.eval()
    output = model(input_tensor)
    pred_class = output.argmax(1)
    output[0, pred_class].backward()
    
    saliency = input_tensor.grad.abs().squeeze().max(dim=0)[0].cpu().numpy()
    
    fig, axes = plt.subplots(1, 2, figsize=(8, 4))
    axes[0].imshow(img.resize((224, 224))); axes[0].set_title('Original'); axes[0].axis('off')
    axes[1].imshow(saliency, cmap='hot'); axes[1].set_title('Saliency Map'); axes[1].axis('off')
    plt.tight_layout()
    plt.show()

compute_saliency(sample_path)

---
## 📐 Section 7: Image Registration (Geometric Extension)

In [ ]:
def align_image(img, target_size=(224, 224)):
    """Simple image alignment using center crop and resize"""
    h, w = img.shape[:2]
    # Center crop to square
    size = min(h, w)
    y, x = (h - size) // 2, (w - size) // 2
    cropped = img[y:y+size, x:x+size]
    # Resize to target
    aligned = cv2.resize(cropped, target_size)
    return aligned

def compare_aligned_images(paths):
    """Compare multiple aligned images side by side"""
    fig, axes = plt.subplots(1, len(paths), figsize=(4*len(paths), 4))
    for i, path in enumerate(paths):
        img = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
        aligned = align_image(img)
        axes[i].imshow(aligned)
        axes[i].set_title(os.path.basename(os.path.dirname(path)).split('-')[-1][:15])
        axes[i].axis('off')
    plt.suptitle('Aligned Dog Images')
    plt.tight_layout()
    plt.show()

# Demo with 3 random breeds
sample_paths = [os.path.join(DATA_PATH, breeds[i], os.listdir(os.path.join(DATA_PATH, breeds[i]))[0]) for i in [0, 30, 60]]
compare_aligned_images(sample_paths)

---
## 🖼️ Section 8: Interactive Breed Prediction

In [ ]:
def predict_breed(image_path):
    """Complete pipeline: Detect -> Classify -> Explain"""
    print('🔍 Detecting dog...')
    cropped, bbox = detect_and_crop_dog(image_path)
    
    if bbox is None:
        print('⚠️ No dog detected, using full image')
        img = Image.open(image_path).convert('RGB')
    else:
        img = Image.fromarray(cropped)
    
    # Classify
    print('🧠 Classifying breed...')
    input_tensor = val_transform(img).unsqueeze(0).to(device)
    model.eval()
    with torch.no_grad():
        output = model(input_tensor)
        probs = torch.softmax(output, dim=1)
        top5_probs, top5_idx = probs.topk(5)
    
    # Display results
    print('\n📊 Top-5 Predictions:')
    for i in range(5):
        breed = class_names[top5_idx[0][i]].split('-')[-1]
        prob = top5_probs[0][i].item() * 100
        bar = '█' * int(prob / 5)
        print(f'   {i+1}. {breed}: {prob:.1f}% {bar}')
    
    # Grad-CAM
    print('\n🔥 Generating Grad-CAM...')
    cam, _ = gradcam.generate(input_tensor, top5_idx[0][0].item())
    cam_resized = cv2.resize(cam, (224, 224))
    
    img_np = np.array(img.resize((224, 224))) / 255.0
    heatmap = plt.cm.jet(cam_resized)[:, :, :3]
    overlay = 0.5 * img_np + 0.5 * heatmap
    
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(cv2.cvtColor(cv2.imread(image_path), cv2.COLOR_BGR2RGB))
    if bbox:
        rect = plt.Rectangle((bbox[0], bbox[1]), bbox[2]-bbox[0], bbox[3]-bbox[1], 
                              fill=False, color='lime', linewidth=2)
        axes[0].add_patch(rect)
    axes[0].set_title('Input + Detection'); axes[0].axis('off')
    axes[1].imshow(overlay); axes[1].set_title('Grad-CAM Overlay'); axes[1].axis('off')
    axes[2].imshow(cam_resized, cmap='jet'); axes[2].set_title('Attention Map'); axes[2].axis('off')
    plt.tight_layout()
    plt.show()

print('✅ Prediction function ready!')

In [ ]:
# Upload and predict
print('📤 Upload a dog image to classify:')
uploaded = files.upload()

for filename in uploaded.keys():
    print(f'\n Processing: {filename}')
    predict_breed(filename)

---
## 📈 Summary: Module 2 vs Module 3

| Metric | Module 2 (Classical) | Module 3 (Deep Learning) |
|--------|---------------------|-------------------------|
| Breeds | 50 | 120 |
| Features | HOG + LBP | CNN (Learned) |
| Accuracy | 6-10% | 70-85% |
| Detection | None | YOLOv8 |
| Explainability | None | Grad-CAM + Saliency |